# Standard Deep Neural Network (DNN) - Cancer Risk Dataset

This notebook trains a standard (non-private) DNN classifier on the Cancer Risk dataset.

## 1. Import Libraries

In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print(f"Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Shared metric collector (repository root): logs training accuracy,
# balanced accuracy and AUROC alongside the metrics already tracked below.
import os as _os
import sys as _sys
_REPO_ROOT = _os.path.abspath(_os.path.join('..', '..'))
if _REPO_ROOT not in _sys.path:
    _sys.path.insert(0, _REPO_ROOT)
from metrics_utils import ExtraMetrics


Using CPU
PyTorch version: 2.11.0
Device: cpu


## 2. Load Preprocessed Data

In [2]:
with open("../data/processed_data.pkl", "rb") as f:
    loaded_data = pickle.load(f)

X_train = loaded_data["X_train"]
X_test = loaded_data["X_test"]
y_train = loaded_data["y_train"]
y_test = loaded_data["y_test"]

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Number of features: {X_train.shape[1]}")
print(f"\nClass distribution in training set:")
print(pd.Series(y_train).value_counts())

X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

batch_size = 32
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nData loaded and prepared")

Training set: (1200, 8)
Test set: (300, 8)
Number of features: 8

Class distribution in training set:
0    754
1    446
Name: count, dtype: int64

Data loaded and prepared


## 3. Define DNN Architecture

In [3]:
class DNNClassifier(nn.Module):
    def __init__(self, input_size=8, hidden_sizes=[64, 32], num_classes=2, dropout=0.3):
        super(DNNClassifier, self).__init__()
        layers = []
        prev_size = input_size
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size
        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

input_size = X_train.shape[1]
hidden_sizes = [64, 32]
num_classes = 2
dropout = 0.3
learning_rate = 0.001
num_epochs = 50

print(f"Model Architecture:")
print(f"  Input: {input_size} features")
print(f"  Hidden layers: {hidden_sizes}")
print(f"  Output: {num_classes} classes")
print(f"  Dropout: {dropout}")
print(f"  Learning rate: {learning_rate}")
print(f"  Epochs: {num_epochs}")

Model Architecture:
  Input: 8 features
  Hidden layers: [64, 32]
  Output: 2 classes
  Dropout: 0.3
  Learning rate: 0.001
  Epochs: 50


## 4. Training Functions

In [4]:
def train_model(model, train_loader, criterion, optimizer, num_epochs, device, verbose=False):
    model.train()
    train_losses = []
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)
        if verbose and (epoch + 1) % 10 == 0:
            print(f"  Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")
    return train_losses

def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

print("Training functions defined")

Training functions defined


## 5. Train Standard DNN (30 Runs)

In [5]:
N_RUNS = 30

detail_accuracies = []
detail_f1s = []
detail_precisions = []
detail_recalls = []
detail_roc_auc = []
detail_cms = []

print(f"Training Standard DNN {N_RUNS} times...")
print("="*80)

extra = ExtraMetrics()
for run in range(N_RUNS):
    seed = run * 10 + 42
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    model = DNNClassifier(input_size, hidden_sizes, num_classes, dropout).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    _t0 = time.perf_counter()
    train_losses = train_model(model, train_loader, criterion, optimizer, num_epochs, device)
    train_time = time.perf_counter() - _t0
    # Training-partition score for the non-private fit.
    y_train_true, y_train_pred, _ = evaluate_model(model, train_loader, device)
    y_true, y_pred, y_prob = evaluate_model(model, test_loader, device)

    accuracy = accuracy_score(y_true, y_pred)
    extra.add(y_true, y_pred, y_score=y_prob, train_true=y_train_true, train_pred=y_train_pred, train_time=train_time)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_true, y_prob)
    cm = confusion_matrix(y_true, y_pred)

    detail_accuracies.append(accuracy)
    detail_f1s.append(f1)
    detail_precisions.append(precision)
    detail_recalls.append(recall)
    detail_roc_auc.append(roc_auc)
    detail_cms.append(cm)

    print(f"  Run {run+1:2d}/{N_RUNS} | Acc: {accuracy:.4f} | F1: {f1:.4f} | Prec: {precision:.4f} | Rec: {recall:.4f}")

print("="*80)
print("Training complete")

Training Standard DNN 30 times...


  Run  1/30 | Acc: 0.8667 | F1: 0.8131 | Prec: 0.8447 | Rec: 0.7838


  Run  2/30 | Acc: 0.8867 | F1: 0.8440 | Prec: 0.8598 | Rec: 0.8288


  Run  3/30 | Acc: 0.8867 | F1: 0.8426 | Prec: 0.8667 | Rec: 0.8198


  Run  4/30 | Acc: 0.8867 | F1: 0.8426 | Prec: 0.8667 | Rec: 0.8198


...

  Run 26/30 | Acc: 0.8900 | F1: 0.8493 | Prec: 0.8611 | Rec: 0.8378


  Run 27/30 | Acc: 0.8833 | F1: 0.8387 | Prec: 0.8585 | Rec: 0.8198


  Run 28/30 | Acc: 0.8767 | F1: 0.8326 | Prec: 0.8364 | Rec: 0.8288


  Run 29/30 | Acc: 0.8867 | F1: 0.8426 | Prec: 0.8667 | Rec: 0.8198


  Run 30/30 | Acc: 0.8900 | F1: 0.8479 | Prec: 0.8679 | Rec: 0.8288
Training complete


In [6]:
# Export model for MIA (use the last trained model)
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..', '..')))
from model_exporter import export_model

export_model(model, 'output/model/std_dnn_model.pkl')

PyTorch model successfully exported to output/model/std_dnn_model.pkl


## 6. Evaluation Results

In [7]:
accuracy = float(np.mean(detail_accuracies))
f1 = float(np.mean(detail_f1s))
precision = float(np.mean(detail_precisions))
recall = float(np.mean(detail_recalls))
roc_auc = float(np.mean(detail_roc_auc))
std_accuracy = float(np.std(detail_accuracies))
std_f1 = float(np.std(detail_f1s))

print("=" * 60)
print("STANDARD DNN - EVALUATION METRICS (CANCER RISK DATASET)")
print("=" * 60)
print(f"Runs:      {N_RUNS}")
print(f"Accuracy:  {accuracy:.4f} +/- {std_accuracy:.4f}")
print(f"F1-Score:  {f1:.4f} +/- {std_f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print("=" * 60)

STANDARD DNN - EVALUATION METRICS (CANCER RISK DATASET)
Runs:      30
Accuracy:  0.8832 +/- 0.0064
F1-Score:  0.8394 +/- 0.0091
Precision: 0.8546
Recall:    0.8249
ROC-AUC:   0.9475


## 7. Save Baseline Results

In [8]:
import os
os.makedirs('output', exist_ok=True)

baseline_results = {
    'model': 'Standard DNN',
    'dataset': 'Cancer Risk Prediction',
    'architecture': str(hidden_sizes),
    'input_features': input_size,
    'accuracy': accuracy,
    'f1_score': f1,
    'precision': precision,
    'recall': recall,
    'roc_auc': roc_auc,
    'accuracy_std': std_accuracy,
    'f1_std': std_f1
}
baseline_results.update(extra.means())

baseline_df = pd.DataFrame([baseline_results])
baseline_df.to_csv('output/baseline_dnn_results.csv', index=False)

results_json = {
    "model_type": "Standard DNN",
    "dataset": "Cancer Risk Prediction",
    "accuracy": accuracy,
    "accuracy_std": std_accuracy,
    "f1_score": f1,
    "f1_std": std_f1,
    "precision": precision,
    "recall": recall,
    "roc_auc": roc_auc,
    "parameters": {
        "architecture": "DNN",
        "input_size": input_size,
        "hidden_sizes": hidden_sizes,
        "num_classes": num_classes,
        "dropout": dropout,
        "learning_rate": learning_rate,
        "num_epochs": num_epochs,
        "batch_size": batch_size,
        "optimizer": "Adam",
        "n_runs": N_RUNS
    }
}
results_json.update(extra.means())

with open('output/std_dnn_report.json', 'w') as f:
    json.dump(results_json, f, indent=4)

print("Results saved to 'output/baseline_dnn_results.csv' and 'output/std_dnn_report.json'")

print("Added metrics:")
print(extra.describe())

Results saved to 'output/baseline_dnn_results.csv' and 'output/std_dnn_report.json'
Added metrics:
  Train Acc:      0.9047 ± 0.0030
  Balanced Acc:   0.8712 ± 0.0074
  AUROC:          0.9475 ± 0.0016
  Train Time (s): 0.5033 ± 0.0957


## 8. Summary

In [9]:
print("\n" + "="*60)
print("BASELINE MODEL SUMMARY")
print("="*60)
print(f"Model: Standard DNN")
print(f"Architecture: Input({input_size}) -> Dense(64,ReLU) -> Dropout(0.3) -> Dense(32,ReLU) -> Dropout(0.3) -> Output(2)")
print(f"Dataset: Cancer Risk Prediction ({X_train.shape[0]:,} train, {X_test.shape[0]:,} test)")
print(f"\nPerformance Metrics ({N_RUNS} runs):")
print(f"  Accuracy:  {accuracy:.4f} +/- {std_accuracy:.4f}")
print(f"  F1-Score:  {f1:.4f} +/- {std_f1:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  ROC-AUC:   {roc_auc:.4f}")
print(f"\nBaseline established for DP-DNN comparison")
print("="*60)


BASELINE MODEL SUMMARY
Model: Standard DNN
Architecture: Input(8) -> Dense(64,ReLU) -> Dropout(0.3) -> Dense(32,ReLU) -> Dropout(0.3) -> Output(2)
Dataset: Cancer Risk Prediction (1,200 train, 300 test)

Performance Metrics (30 runs):
  Accuracy:  0.8832 +/- 0.0064
  F1-Score:  0.8394 +/- 0.0091
  Precision: 0.8546
  Recall:    0.8249
  ROC-AUC:   0.9475

Baseline established for DP-DNN comparison
